# 01 - Train / Test Split

## Objetivo

Partimos de un dataset de LendingClub con **~1.76 millones** de préstamos y 151 variables. El objetivo de este notebook es generar **3 datasets** para usar en posteriores modelos de Machine Learning:

| Dataset | Filas | Distribución de `loan_status` | Uso |
|---|---|---|---|
| `df_train_small` | 80.000 | ~80/20 (original, desbalanceada) | Entrenar modelo A |
| `df_train_balanced` | 80.000 | 50/50 (balanceada) | Entrenar modelo B |
| `df_test_small` | 20.000 | ~80/20 (original, desbalanceada) | Evaluar ambos modelos |

Tener dos sets de entrenamiento (uno desbalanceado y otro balanceado) nos permitirá comparar cómo afecta el balanceo de clases al rendimiento predictivo.

In [1]:
import pandas as pd


## Paso 1 — Carga del dataset original

Se carga el CSV completo de LendingClub (préstamos aceptados entre 2007 y 2017). El resultado es un DataFrame de **~1.76 millones de filas** y **151 columnas**.

In [2]:
df = pd.read_csv("data/accepted_2007_to_2017.csv")
df.shape

/var/folders/90/nx4mc_9d0wv10t5rvcldvb940000gn/T/ipykernel_25725/2270568562.py:1: DtypeWarning: Columns (0: desc, 1: next_pymnt_d, 2: verification_status_joint, 3: sec_app_earliest_cr_line, 4: hardship_type, 5: hardship_reason, 6: hardship_status, 7: hardship_start_date, 8: hardship_end_date, 9: payment_plan_start_date, 10: hardship_loan_status) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/accepted_2007_to_2017.csv")


(1765426, 151)

## Paso 2 — Exploración de la variable objetivo

Inspeccionamos la distribución de `loan_status`. Existen 9 categorías distintas, pero muchas son **estados intermedios** (Current, In Grace Period, Late...) que no representan un desenlace final del préstamo. Solo nos interesan los estados terminales: **Fully Paid** (pagado) y **Charged Off** (impago).

In [3]:
df["loan_status"].value_counts(normalize=True)*100

loan_status
Fully Paid                                             58.303605
Current                                                25.553946
Charged Off                                            14.709877
Late (31-120 days)                                      0.806944
In Grace Period                                         0.312502
Late (16-30 days)                                       0.155543
Does not meet the credit policy. Status:Fully Paid      0.112607
Does not meet the credit policy. Status:Charged Off     0.043106
Default                                                 0.001869
Name: proportion, dtype: float64

## Paso 3 — Filtrado: solo préstamos con desenlace definitivo

Nos quedamos únicamente con los préstamos que tienen un resultado final:
- **Fully Paid** (~80%): el prestatario devolvió el préstamo correctamente.
- **Charged Off** (~20%): el prestatario incumplió (impago).

Esto reduce el dataset a **~1.29 millones de filas** con una distribución **desbalanceada ~80/20**, que refleja la realidad del negocio: la mayoría de los préstamos se pagan.

In [4]:
# filtrar solo full y paid y charged off
df_filtered = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]


In [5]:
df_filtered.shape

(1288999, 151)

In [6]:
df_filtered["loan_status"].value_counts(normalize=True) * 100

loan_status
Fully Paid     79.853204
Charged Off    20.146796
Name: proportion, dtype: float64

## Paso 4 — Split train/test (80/20)

Dividimos el dataset filtrado (~1.29M filas) en dos conjuntos usando `train_test_split` de scikit-learn:
- **Train** (~1.03M filas): datos para entrenar los modelos.
- **Test** (~258K filas): datos reservados para evaluar los modelos.

Se usa `random_state=42` para garantizar la **reproducibilidad**: cada vez que ejecutemos el notebook obtendremos exactamente la misma partición.

In [7]:
# train test split
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df_filtered, test_size=0.2, random_state=42)

## Paso 5 — Submuestreo de test: 20.000 filas

Del conjunto de test (~258K filas) extraemos una muestra aleatoria de **20.000 filas** para trabajar con un tamaño manejable.

**¿Por qué test NO se balancea?** Porque el conjunto de test representa los **datos no vistos** que simulan el mundo real. Si alterásemos su distribución, estaríamos evaluando el modelo en un escenario ficticio. El test **siempre debe reflejar la distribución original** (~80/20) para que las métricas (accuracy, precision, recall, F1...) sean representativas de cómo se comportará el modelo en producción.

In [8]:
df_test_small = df_test.sample(20_000, random_state=42)

In [9]:
df_test_small["loan_status"].value_counts(normalize=True) * 100

loan_status
Fully Paid     80.015
Charged Off    19.985
Name: proportion, dtype: float64

## Paso 6 — Submuestreo de train desbalanceado: 80.000 filas

Del conjunto de train (~1.03M filas) extraemos **80.000 muestras** aleatorias. Este muestreo conserva la distribución original ~80/20 ya que es una muestra aleatoria simple.

Este será el **primer set de entrenamiento** (desbalanceado), que refleja la proporción real de pagos e impagos.

In [10]:
df_train_small = df_train.sample(80_000, random_state=42)

In [11]:
df_train_small["loan_status"].value_counts(normalize=True) * 100

loan_status
Fully Paid     79.705
Charged Off    20.295
Name: proportion, dtype: float64

## Paso 7 — Creación del train balanceado: 80.000 filas (50/50)

Construimos el **segundo set de entrenamiento** con distribución **50/50** mediante **undersampling** (submuestreo de la clase mayoritaria):

1. Se toman **40.000** muestras aleatorias de "Fully Paid" del train.
2. Se toman **40.000** muestras aleatorias de "Charged Off" del train.
3. Se concatenan ambas muestras.
4. Se **barajan** las filas (`sample(frac=1)`) para que no queden ordenadas por clase, lo cual podría introducir sesgos durante el entrenamiento.

El objetivo es que el modelo vea **la misma cantidad de ejemplos de cada clase** y aprenda mejor a detectar los impagos, que es la clase minoritaria pero la más importante desde el punto de vista del negocio.

In [12]:
# crear una muestra de train balanceada con respecto a loan_status
df_train_full_paid = df_train[df_train["loan_status"] == "Fully Paid"].sample(40_000, random_state=42)
df_train_charged_off = df_train[df_train["loan_status"] == "Charged Off"].sample(40_000, random_state=42)
df_train_balanced = pd.concat([df_train_full_paid, df_train_charged_off]) 

df_train_balanced = df_train_balanced.sample(frac=1, random_state=42)

In [13]:
df_train_balanced["loan_status"].value_counts(normalize=True) * 100

loan_status
Charged Off    50.0
Fully Paid     50.0
Name: proportion, dtype: float64

## Paso 8 — Guardado de los 3 datasets finales

Exportamos los tres DataFrames a CSV:

| Archivo | Contenido | Filas | Distribución |
|---|---|---|---|
| `df_train_small.csv` | Train desbalanceado | 80.000 | ~80/20 |
| `df_train_balanced.csv` | Train balanceado | 80.000 | 50/50 |
| `df_test_small.csv` | Test | 20.000 | ~80/20 |

En los siguientes notebooks entrenaremos modelos con cada set de train y los evaluaremos sobre **el mismo test**, para comparar el efecto del balanceo de clases en el rendimiento predictivo.

In [14]:
# save df_train_balanced, df_test_small and df_train_small to csv
df_train_balanced.to_csv("data/df_train_balanced.csv", index=False) 
df_test_small.to_csv("data/df_test_small.csv", index=False) 
df_train_small.to_csv("data/df_train_small.csv", index=False)